# Week 2. Counting Is Already a Model

**What you'll do today.** Build a word-counter by hand-logic and watch tokenizing,
stemming, and stop lists change its answer, all on live Reddit. Count phrases (n-grams),
then run the week's featured trial (Bollen v. Schmidt) with its own data, pulled live
from Google Books. Pick up the basic statistics of counts: distributions, rates, and
what a mean hides. Use tf-idf to sight-read twelve communities from their distinctive
words, and k-means to sort comments and paintings alike from counts alone. Upgrade the
signature-vocabulary trick into keyness, and finish with the one statistics move the
course requires: the shuffle test for whether a difference is real.


> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.


In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# Fetch the pinned package lists if they aren't beside the notebook (this happens
# whenever you open a single notebook straight from GitHub in Colab).
import os, urllib.request
_RAW = "https://raw.githubusercontent.com/lucianli123/culture-as-data-2026/main/notebooks/"
for _f in ("requirements.txt", "constraints.txt"):
    if not os.path.exists(_f):
        urllib.request.urlretrieve(_RAW + _f, _f)

# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below installs them:
%pip install -q -r requirements.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q pandas scikit-learn matplotlib pillow
# If an import later fails with "numpy.dtype size changed" or similar: the install
# replaced a compiled library under an already-loaded module. Runtime -> Restart
# session, then run again from the top. It should not recur.


In [ ]:
# Imports. If this cell fails, it almost always means the install cell above didn't run
# this session, re-run it, then re-run this. (See the cheat sheet, entry 5.)
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
print("imports ok")

In [ ]:
import os

## Three real communities

The week's corpus: comments from three subreddits with three vocabularies, pulled live
from the Arctic Shift archive (the Pushshift successor; it covers essentially all of
Reddit). Analyze-only, as always: we publish counts and findings, never the text, and
no usernames.


In [ ]:
import re
import time
import requests

def arctic_sample(sub, n=100, tries=3):
    """n comments from one subreddit, newest first. Nothing is saved or republished.
    Retries politely: a whole classroom hitting the archive at once gets rate-limited."""
    for attempt in range(tries):
        got, before, pages = [], None, 0
        try:
            while len(got) < n and pages < 2 + n // 100:
                params = {"subreddit": sub, "limit": 100, "fields": "body,created_utc"}
                if before: params["before"] = before
                r = requests.get("https://arctic-shift.photon-reddit.com/api/comments/search",
                                 params=params, timeout=30)
                r.raise_for_status()
                rows = r.json()["data"]
                pages += 1
                if not rows: break
                before = int(min(x["created_utc"] for x in rows))
                got += [d["body"] for d in rows if isinstance(d.get("body"), str)
                        and 80 < len(d["body"]) < 800
                        and "[removed]" not in d["body"] and "[deleted]" not in d["body"]
                        and "I am a bot" not in d["body"]]
            if got:
                return got[:n]
            print(f"r/{sub}: empty response (attempt {attempt + 1}/{tries}), waiting a moment...")
        except Exception as e:
            print(f"r/{sub}: {type(e).__name__} (attempt {attempt + 1}/{tries}), waiting a moment...")
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(
        f"r/{sub} returned nothing after {tries} tries. This is usually the archive "
        "rate-limiting a busy classroom: wait a minute, then rerun this cell. "
        "Also check the subreddit name is spelled exactly (case matters).")

SUBS = ["buildapc", "gardening", "relationship_advice"]   # three worlds
corpora = {sub: arctic_sample(sub, n=300) for sub in SUBS}   # ~30 seconds
for name, docs_ in corpora.items():
    print(f"{name:22s} {len(docs_):4d} comments |", docs_[0][:60], "...")


## Bag-of-words by hand: counting *requires defining*

Before any library, count by hand-logic.

In [ ]:
def tokenize(text, fold_case=True, strip_punct=True):
    if fold_case:
        text = text.lower()
    if strip_punct:
        text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [t for t in text.split() if t]

sample_text = " ".join(corpora["relationship_advice"][:60])
counts = Counter(tokenize(sample_text))
print("Top words in r/relationship_advice:")
for word, n in counts.most_common(8):
    print(f"  {word:8s} {n}")


In [ ]:
# A "stop word" list is itself a choice. Watch the top-words list change when we drop
# common function words, the same kind of decision as merging run/running.
STOP = {"i","we","you","it","the","a","and","to","of","in","so","be","on","my","me","now",
        "is","that","this","was","have","for","with","are","but","he","she","they","your"}
counts_nostop = Counter(t for t in tokenize(sample_text) if t not in STOP)
print("Top words with stop-words removed:")
for word, n in counts_nostop.most_common(8):
    print(f"  {word:8s} {n}")
# Still noisy? Informal English has its own function words (just, like, dont, really)
# that no standard list includes. The stop list must fit the corpus in front of you;
# the next cell makes that point with a professional list.


### What counts as a word? Tokens, not words

Models never see words.

## Stemming, lemmatization, and a real stop list

`tokenize` split the text. It did not decide that "run", "runs", and "running" are the
same word. Deciding that is **standardization**: collapsing variation you consider
irrelevant so that counts compare fairly, the same move a z-score makes for numbers
(more on those in the statistics section below). Two standard tools make that call, and they disagree. A **stemmer** chops
endings by rule. A **lemmatizer** looks the word up and returns its dictionary form.
NLTK ships both, plus a stop list far longer than our hand-built one. (Colab has NLTK
preinstalled; elsewhere, `pip install nltk` first.)


In [ ]:
# needs: pip install nltk  (already on Colab)
try:
    import nltk
    for pkg in ["stopwords", "wordnet", "omw-1.4"]:
        nltk.download(pkg, quiet=True)
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer, WordNetLemmatizer

    NLTK_STOP = set(stopwords.words("english"))
    print(f"NLTK's English stop list: {len(NLTK_STOP)} words. Our hand list: {len(STOP)}.")
    fillers = ["just", "like", "really", "actually", "basically", "lol", "im", "dont", "thats"]
    print("Reddit's own function words, missing from NLTK's list:",
          [w for w in fillers if w not in NLTK_STOP])
    # Even a professional list misses this corpus's fillers. No stop list is neutral,
    # and none is finished: it must fit the corpus in front of you.

    stem = PorterStemmer().stem
    lemma = WordNetLemmatizer().lemmatize

    # No invented examples: harvest the corpus's own word families. Group every word
    # in r/gardening by its stem and take the families with the most surface forms.
    tokens = tokenize(" ".join(corpora["gardening"]))
    from collections import defaultdict
    families = defaultdict(set)
    for t in set(tokens):
        if len(t) > 3:
            families[stem(t)].add(t)
    real_forms = sorted(sorted(families.values(), key=len, reverse=True)[0] |
                        sorted(families.values(), key=len, reverse=True)[1])

    print(f"\none corpus, one stem, {len(real_forms)} surface forms:")
    print(f"{'word':14s} {'stem':12s} {'lemma (as noun)':16s} {'lemma (as verb)':16s}")
    for w in real_forms:
        print(f"{w:14s} {stem(w):12s} {lemma(w):16s} {lemma(w, 'v'):16s}")
    # Read the stem column: the stemmer's output is often not a word, and it can
    # lump forms you might want apart. Read the lemma columns: right, but only when
    # told the part of speech. Both are choices about sameness, made on YOUR data.

    print(f"\nr/gardening vocabulary: {len(set(tokens))} raw, "
          f"{len({stem(t) for t in tokens})} stemmed, "
          f"{len({lemma(t) for t in tokens})} lemmatized")
    # Merging forms shrinks the vocabulary and fattens each count. Whether they are
    # one word or many is your call, and it changes every chart downstream.
except Exception as e:
    print("NLTK unavailable here, skipping this demo:", e)


In [ ]:
# spaCy tags parts of speech and lemmatizes in one pass, so it does not need telling.
# Run it on a real comment from the corpus you just loaded.
# Preinstalled on Colab; elsewhere: pip install spacy, then download en_core_web_sm.
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    comment = next((c for c in corpora["gardening"] if len(c) < 200), corpora["gardening"][0])
    print(comment[:120], "\n")
    print(f"{'token':14s} {'part of speech':14s} lemma")
    for t in nlp(comment)[:14]:
        print(f"{t.text:14s} {t.pos_:14s} {t.lemma_}")
    # Where NLTK guessed noun-unless-told, spaCy reads the sentence first: the same
    # surface form gets the right lemma from context. The price is a model, not a
    # rule book: slower, heavier, and occasionally confidently wrong. Rerun this cell
    # on a different comment and look for one mistake.
except Exception as e:
    print("spaCy not available here; the NLTK cell above covers the lesson.", e)


## Zipf's law: every corpus has the same shape

Rank the words by frequency and plot count against rank, both on log scales. Any corpus
in any language gives roughly a straight line: the second word is about half as common
as the first, the tenth about a tenth as common. A handful of words does most of the
work, and thousands appear once.


In [ ]:
plt.figure(figsize=(7, 4))
top_count = 0
for name, docs_ in corpora.items():
    freqs = sorted(Counter(tokenize(" ".join(docs_))).values(), reverse=True)
    top_count = max(top_count, freqs[0])
    plt.loglog(range(1, len(freqs) + 1), freqs, label=f"{name} ({len(freqs):,} distinct words)")
ranks = np.arange(1, 10_000)
plt.loglog(ranks, top_count / ranks, "k--", linewidth=1, label="1/rank reference")
plt.xlabel("word rank"); plt.ylabel("count"); plt.legend()
plt.title("Zipf's law: three corpora, one shape")
plt.tight_layout(); plt.show()

# The head of the curve is the stop list: the, and, of are so common they swamp any
# count that keeps them. The tail is the other warning: most words are too rare to
# count reliably, which is part of why Week 5 replaces words with dimensions.


## N-grams at corpus scale

Week 1 counted word pairs in one corpus. The same move, across the three communities:
the top bigrams are each subreddit's fingerprint.


In [ ]:
def bigram_counts(docs_, stop=()):
    c = Counter()
    for d in docs_:
        ws = tokenize(d)
        for a, b in zip(ws, ws[1:]):
            if a not in stop and b not in stop:
                c[a + " " + b] += 1
    return c

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, (name, docs_) in zip(axes, corpora.items()):
    top = bigram_counts(docs_, STOP).most_common(8)
    ax.barh([p for p, _ in top][::-1], [n for _, n in top][::-1], color="#3f6f5f")
    ax.set_title(name, fontsize=10)
fig.suptitle("Top bigrams per corpus (stop words filtered)", y=1.04)
plt.tight_layout(); plt.show()
# Pairs carry what single words lose: "power supply" and "long distance" are things,
# not word pairs. And the leftover filler pairs are the stop-list lesson again, at
# phrase scale: no list is ever quite done.


## The trial, live: Bollen v. Schmidt

The week's featured exchange is an n-gram study. Bollen et al. (PNAS 2021) counted
cognitive-distortion phrases ("I am a failure") in Google Books and found a hockey
stick after 2000; Schmidt et al. replied that the corpus itself changed, with more
fiction scanned in recent years. The Ngram Viewer has a JSON endpoint, so the evidence
is one request away. This is a probe, not a replication: the paper counts 241 phrases,
we pull two. (If you get HTTP 429, the endpoint is rate-limited; wait a minute.)


In [ ]:
import requests

def google_ngrams(phrases, corpus="en-2019", start=1900, end=2019):
    """Yearly share of each phrase in Google Books, straight from the Ngram Viewer."""
    r = requests.get("https://books.google.com/ngrams/json",
                     params={"content": ",".join(phrases), "year_start": start,
                             "year_end": end, "corpus": corpus, "smoothing": 3},
                     timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    return pd.DataFrame({s["ngram"]: s["timeseries"] for s in r.json()},
                        index=range(start, end + 1))

claim = google_ngrams(["I am a failure", "everyone hates me"])
claim.plot(figsize=(8, 3), color=["#7a3b2e", "#b9852f"])
plt.title("The claim, reproduced live: distortion phrases in Google Books")
plt.ylabel("share of all bigrams/n-grams"); plt.tight_layout(); plt.show()
# There is the hockey stick. Now the objection.


In [ ]:
# The objection, counted. If the corpus quietly gained fiction, phrases that mark
# fiction should hockey-stick too, with no change in anyone's mind:
suspects = google_ngrams(["she whispered", "he grinned"])
suspects.plot(figsize=(8, 3), color=["#3f6f5f", "#2e5f8a"])
plt.title("Fiction markers rise the same way: the corpus changed under the count")
plt.tight_layout(); plt.show()

# The sharpest single check: the same phrase inside fiction only, against all books.
both = pd.DataFrame({
    "all books": google_ngrams(["I am a failure"]).iloc[:, 0],
    "fiction only": google_ngrams(["I am a failure"], corpus="en-fiction-2019").iloc[:, 0],
})
both.plot(figsize=(8, 3), color=["#7a3b2e", "#6d4b7c"])
plt.title('"I am a failure": flat within fiction, rising in the mixed corpus')
plt.tight_layout(); plt.show()
# Within fiction the phrase is roughly level for a century; the rise lives in the mixed
# corpus, which is what you would see if composition did the work. The authors' rebuttal
# removed fiction and reported the pattern largely held, so the trial is not over.
# Discussion: what further count would change your verdict?


## The statistics of counts

Before comparing counts you need four habits: look at the distribution (not just the
mean), divide by a denominator, put values on a common scale, and only then ask if a
gap is real.


In [ ]:
# Habit 1: look at the distribution. "plant" per r/gardening comment:
plant_per = pd.Series([len(re.findall(r"\bplant\w*\b", c.lower())) for c in corpora["gardening"]])
print(plant_per.describe().round(2))
plant_per.hist(bins=range(0, plant_per.max() + 2), color="#3f6f5f", figsize=(6, 2.6))
plt.title('"plant*" per comment in r/gardening'); plt.xlabel("count in a comment")
plt.ylabel("comments"); plt.tight_layout(); plt.show()
# Mean and median disagree because count data is skewed: most comments mention plants
# zero or one time, a few mention many. That is the normal shape of counts; a lone
# mean hides it.


In [ ]:
# Habit 2: divide by a denominator. Communities differ in how much they write, so raw
# totals mislead. Rates per 1,000 words compare fairly.
def per_1000(docs_, word):
    hits = sum(len(re.findall(rf"\b{word}\b", d.lower())) for d in docs_)
    return 1000 * hits / sum(len(tokenize(d)) for d in docs_)

for w in ["water", "money", "feel"]:
    print(f"{w:6s}", {name: round(per_1000(dd, w), 2) for name, dd in corpora.items()})
# Read these three rows skeptically; they are not equally trustworthy.
#   "water": a huge gap. Hard to fake.
#   "money": a small gap between buildapc and advice, and it flips sign from
#            one pull to the next. Difference, or dice roll?
#   a 0.0: never trust a zero from a small sample; one comment flips it.
# Habit 3, next cell: put unlike numbers on a common scale. And habit 4: a gap can
# still be luck. The "money" row comes back at the end of the notebook, where the
# shuffle test settles it.


In [ ]:
# Habit 4 (standardize): "is 5 a lot?" has no answer until you know what's typical.
# A z-score rescales any number as "how many standard deviations from the mean":
#     z = (value - mean) / std
plant_counts = pd.Series([len(re.findall(r"\bplant\w*\b", c.lower())) for c in corpora["gardening"]])
lengths = pd.Series([len(tokenize(c)) for c in corpora["gardening"]])

one = int(plant_counts.idxmax())     # the plant-heaviest comment (try any number)
for name, series in [("plant mentions", plant_counts), ("comment length", lengths)]:
    z = (series[one] - series.mean()) / series.std()
    print(f"comment #{one}, {name:15s} value={series[one]:5.0f}   z={z:+.2f}")
# A z near 0 is ordinary; past +2, notably unusual; past +3, rare. One caution: those
# thresholds assume roughly symmetric data, and habit 1 just showed you counts are
# not. On zero-heavy counts, z exaggerates; the histogram is the referee. Still:
# mentions and word-lengths, two different scales, are now both "how unusual, for
# this corpus," and comparable. That is standardization, and you have
# met it twice already without the name. Lemmatization standardizes word FORMS
# (running, ran -> run) so counts compare fairly; the shuffle test standardizes a gap
# against the spread of chance gaps. Same idea: remove irrelevant variation, then compare.


## tf-idf: "common here, rare overall"

Raw counts are dominated by words that are common *everywhere* (*the*, *you*).

In [ ]:
docs = corpora["buildapc"] + corpora["gardening"] + corpora["relationship_advice"]
labels = (["buildapc"] * len(corpora["buildapc"]) + ["gardening"] * len(corpora["gardening"])
          + ["relationship_advice"] * len(corpora["relationship_advice"]))
vec = TfidfVectorizer(stop_words="english")
X = vec.fit_transform(docs)
terms = np.array(vec.get_feature_names_out())

# The most distinctive terms of three individual comments, one from each community:
for i in [0, len(corpora["buildapc"]), len(corpora["buildapc"]) + len(corpora["gardening"])]:
    row = X[i].toarray().ravel()
    print(f"{labels[i]:22s}", ", ".join(terms[row.argsort()[::-1][:4]]))
# "Common here, rare overall": tf-idf finds what marks a document out from the pile.


### tf-idf at community scale: what marks each subreddit out

Change what counts as a document and tf-idf changes what it finds. Single comments as
documents gave each comment's signature. Now: one document per *community*. Paste a
sample of each of twelve subreddits into one long document each, and "common here, rare
overall" becomes "what this community talks about that the others don't."


In [ ]:
# arctic_sample is defined at the top of the notebook.
SUBS12 = ["askscience", "nba", "Cooking", "personalfinance", "relationship_advice",
          "gardening", "buildapc", "movies", "Parenting", "cats", "travel", "wallstreetbets"]

def strip_links(t):
    return re.sub(r"http\S+|www\.\S+", " ", t)      # links are not words

sub_docs = {s: strip_links(" ".join(arctic_sample(s))) for s in SUBS12}   # ~15 seconds

vec_s = TfidfVectorizer(stop_words="english", min_df=2, max_df=0.7, sublinear_tf=True)
X_s = vec_s.fit_transform(sub_docs.values())
terms_s = np.array(vec_s.get_feature_names_out())
for k, name in enumerate(sub_docs):
    row = X_s[k].toarray().ravel()
    print(f"{name:22s}", ", ".join(terms_s[row.argsort()[::-1][:6]]))
# Twelve communities, sight-read from their distinctive words: meals and veggies,
# fans and coolers, flights, kittens, stocks and puts. Two quiet settings did real
# work: max_df=0.7 drops words most communities share (just, like), and the link
# strip keeps URL syntax from posing as vocabulary. Cleaning is part of the method,
# and every cleaning step is a choice to report. Swap in any subreddits and rerun.


## Clustering: three communities, separable from counts alone

You already have the corpus: the same 900 comments. Ask k-means for three piles without
telling it the subreddits, and see whether its piles match.

**How to read the tables below.** Every comment gets two labels. Its **truth** is the
subreddit it actually came from (we know, because we loaded it). Its **pile** is the
group k-means put it in; the numbers 0, 1, 2 are arbitrary names the algorithm invented,
so "pile 1" means nothing until you look at what landed there. Each cell of the table
counts comments: row = truth, column = pile. Perfect separation puts each subreddit's
whole row into one column of its own; failure smears every row evenly across the
columns.


In [ ]:
posts = docs            # the 900 comments from the tf-idf cell
subs_true = labels
print(len(posts), "comments,", len(set(subs_true)), "communities")


In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

# Three representations of the same comments, weakest to strongest.
raw_r = CountVectorizer(stop_words="english", min_df=2).fit_transform(posts)
vec_r = TfidfVectorizer(stop_words="english", min_df=2)
tfidf_r = vec_r.fit_transform(posts)

def cluster_table(M, name):
    piles = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(M)
    print(name)
    print(pd.crosstab(pd.Series(subs_true, name="truth"), pd.Series(piles, name="pile")), "\n")
    return piles

cluster_table(raw_r, "raw counts:")          # one blob: everywhere-words dominate
cluster_table(tfidf_r, "tf-idf:")            # better, still smeared

# Step three: compress the thousands of word-columns into 60 summary dimensions
# (TruncatedSVD is PCA for sparse counts), then cluster. A preview of Week 5's
# whole idea: replace word columns with learned dimensions.
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
Z = normalize(TruncatedSVD(60, random_state=0).fit_transform(tfidf_r))
piles = cluster_table(Z, "tf-idf + 60 dimensions:")
# About four comments in five land with their community, and no one told k-means the
# labels. The representation did the work: same data, same algorithm, three answers.

# What IS each pile, in words? Average the tf-idf rows of a pile's members; the
# highest-weighted words are the vocabulary that pulled those comments together.
terms_r = np.array(vec_r.get_feature_names_out())
for k in range(3):
    centroid = np.asarray(tfidf_r[piles == k].mean(axis=0)).ravel()
    print(f"pile {k}:", ", ".join(terms_r[centroid.argsort()[::-1][:10]]))
# Now the arbitrary pile numbers have faces: one pile is cpu/ram/gpu, one is
# plant/garden/water, and one is relationship talk. The table's stragglers explain
# themselves too: a buildapc comment with no part names is just conversation, and it
# sorts with the talk pile. THIS is how you name a cluster: by its words, never by
# its number.


In [ ]:
from sklearn.decomposition import PCA

xy = PCA(n_components=2).fit_transform(Z)
colors = {"buildapc": "#2e5f8a", "gardening": "#3f6f5f", "relationship_advice": "#7a3b2e"}
plt.figure(figsize=(6.5, 4.5))
for name, col in colors.items():
    pts = xy[[j for j, l in enumerate(subs_true) if l == name]]
    plt.scatter(pts[:, 0], pts[:, 1], s=14, alpha=0.7, label=name, color=col)
plt.legend(); plt.title("360 comments, placed by their word counts")
plt.tight_layout(); plt.show()
# Three communities, three regions, from counting alone. The stragglers are worth
# reading: a gardening comment about a partner, a buildapc comment about budgets.
# The cluster found vocabulary, and vocabulary mostly tracks community. Which
# difference a cluster found is always your claim to defend, not the algorithm's.


### The hard version: one city, two communities

buildapc against gardening is easy mode: different topics, different words. r/sandiego
and r/SanDiegan are two communities discussing the *same* city. Does counting still
separate them?


In [ ]:
# One pair of communities, your choice. The Arctic Shift archive (the Pushshift
# successor) covers essentially all of Reddit. Analyze-only, as always: counts and
# findings, never the text or usernames.
SUB_X, SUB_Y = "sandiego", "SanDiegan"   # swap in any two communities you know

# arctic_sample is defined at the top of the notebook.
sx, sy = arctic_sample(SUB_X, n=200), arctic_sample(SUB_Y, n=200)
hard_true = [SUB_X] * len(sx) + [SUB_Y] * len(sy)
print(len(sx), f"r/{SUB_X},", len(sy), f"r/{SUB_Y} comments")

Xh = normalize(TruncatedSVD(40, random_state=0).fit_transform(
    TfidfVectorizer(stop_words="english", min_df=2).fit_transform(sx + sy)))
piles2 = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xh)
print(pd.crosstab(pd.Series(hard_true, name="truth"), pd.Series(piles2, name="pile")))
# The same map as the three-subreddit picture below, for the hard pair: one smear,
# not two regions. What the table says in numbers, the plot says at a glance.
from sklearn.decomposition import PCA
xy_h = PCA(n_components=2).fit_transform(Xh)
plt.figure(figsize=(6, 4))
for name, col in [(SUB_X, "#7a3b2e"), (SUB_Y, "#2e5f8a")]:
    pts = xy_h[[j for j, l in enumerate(hard_true) if l == name]]
    plt.scatter(pts[:, 0], pts[:, 1], s=14, alpha=0.7, label=f"r/{name}", color=col)
plt.legend(); plt.title("The hard pair, mapped: two communities, one cloud")
plt.tight_layout(); plt.show()

# With the default pair, the table barely beats a coin flip: two communities discussing
# the SAME city hardly separate, while different topics separated easily above.
# Now swap SUB_X and SUB_Y for a pair you know and predict the table before you run.
# Week 3 trains a supervised classifier on exactly this task; even shown the labels,
# the same-city pair only reaches about 60 percent, and the words that earn those
# percentage points are the finding.


### The same move for pictures

Week 1 counted pixels; this week's move works on those counts too. Count every pixel of
a painting into 27 color buckets (dark red, bright blue, ...), and each painting becomes
a row of 27 numbers, exactly like a document's word counts. Then the same k-means, no
labels: do portraits and landscapes sort themselves?


In [ ]:
import io
from PIL import Image

def met_images(query, n=10, department=11):
    """Pull n paintings matching a query from the Met's open-access API (CC0)."""
    base = "https://collectionapi.metmuseum.org/public/collection/v1"
    ids = requests.get(f"{base}/search", timeout=20,
                       params={"hasImages": "true", "departmentId": department, "q": query}).json()["objectIDs"]
    out = []
    for oid in ids:
        if len(out) >= n: break
        obj = requests.get(f"{base}/objects/{oid}", timeout=20).json()
        url = obj.get("primaryImageSmall")
        if not url: continue
        try:
            img = Image.open(io.BytesIO(requests.get(url, timeout=20).content)).convert("RGB")
            img.thumbnail((120, 120)); out.append(img)
        except Exception:
            continue
    return out

def color_counts(img):
    """The bag of pixels: every pixel sorted into 27 color buckets (3 levels per channel)."""
    a = np.asarray(img).reshape(-1, 3) // 86          # each channel -> 0, 1, or 2
    idx = a[:, 0] * 9 + a[:, 1] * 3 + a[:, 2]         # 27 buckets
    return np.bincount(idx, minlength=27) / len(idx)  # shares, so size cancels out

QUERY_A, QUERY_B = "portrait", "landscape"            # swap in your own pair
pics_a, pics_b = met_images(QUERY_A), met_images(QUERY_B)
pics = pics_a + pics_b
pic_true = [QUERY_A] * len(pics_a) + [QUERY_B] * len(pics_b)

Xp = np.array([color_counts(p) for p in pics])
pic_piles = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xp)
# Same reading as the subreddit tables: truth = the search query that found the
# painting, pile = the group k-means invented.
print(pd.crosstab(pd.Series(pic_true, name="truth"), pd.Series(pic_piles, name="pile")))

# Look at what the machine grouped, one row per pile:
fig, axes = plt.subplots(2, max((pic_piles == 0).sum(), (pic_piles == 1).sum()),
                         figsize=(12, 3.2))
for row in range(2):
    members = [p for p, c in zip(pics, pic_piles) if c == row]
    for ax, p in zip(axes[row], members):
        ax.imshow(p)
    for ax in axes[row]:
        ax.axis("off")
    axes[row][0].set_title(f"pile {row}", fontsize=9, loc="left")
plt.tight_layout(); plt.show()
# Roughly three paintings in four sort by genre, from color counts alone: portraits
# run to flesh tones on dark grounds, landscapes to greens and skies. The stragglers
# are the discussion: a brown winter landscape counts like a portrait. Same lesson as
# the subreddits, same algorithm, different bag: counts carry real signal, and the
# machine never knows WHICH signal it found.

# The map version: PCA squeezes 27 color counts to 2 coordinates and each painting
# sits at its own point. Neighbors share a palette, whatever their genre.
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from sklearn.decomposition import PCA
xy_p = PCA(n_components=2).fit_transform(Xp)
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.scatter(xy_p[:, 0], xy_p[:, 1], s=1)
for (x, y), p in zip(xy_p, pics):
    small = p.copy(); small.thumbnail((34, 34))
    ax.add_artist(AnnotationBbox(OffsetImage(np.asarray(small)), (x, y), frameon=False))
ax.set_title("Twenty paintings, placed by their color counts alone")
plt.tight_layout(); plt.show()


## Go further: the same counter, three communities (not scheduled in the session)

One counter, run across all three corpora. The corpus choice changes what "common"
means. It also works as a sampler before Week 4: whatever community you bring, this
is the first cell to run on it.


In [ ]:
for name, dd in corpora.items():
    text = " ".join(dd)
    top = Counter(t for t in tokenize(text) if t not in STOP).most_common(5)
    print(f"{name:8s}:", ", ".join(f"{w}({n})" for w, n in top))

### A signature vocabulary

Counting alone can show you a *voice*: the words one source uses far more than a baseline.

In [ ]:
# Words r/gardening uses far more than the other two, by simple rate ratio.
target, rest = corpora["gardening"], corpora["buildapc"] + corpora["relationship_advice"]
tc, rc = Counter(tokenize(" ".join(target))), Counter(tokenize(" ".join(rest)))
NT, NR = sum(tc.values()), sum(rc.values())
def ratio(w):
    return (tc[w] / NT) / ((rc[w] + 1) / NR)
candidates = [w for w, n in tc.items() if n >= 8 and w not in STOP and len(w) > 3]
for w in sorted(candidates, key=ratio, reverse=True)[:10]:
    print(f"{w:14s} {ratio(w):6.1f}x")
# A first cut at "distinctive": how many times likelier is this word here than
# elsewhere? Keyness, next, does this properly (smoothed, symmetric, in log-odds).


## Keyness: distinctively hers, done properly

The ratio above was a rough cut. The standard tool is a **log-odds ratio**: for every word,
how much more likely is it in corpus A than corpus B (with a little smoothing so rare words
don't explode)? Positive means "distinctively A," negative "distinctively B."

**The reveal:** this is *exactly* the method behind Week 1's featured study. *She Giggles, He
Gallops* is a log-odds ratio of stage-direction verbs after "she" vs. "he," run on 2,000
screenplays. You now own the arithmetic behind the piece you admired.

In [ ]:
import numpy as np

def counts(docs_):
    return Counter(w for d in docs_ for w in tokenize(d))

A = counts(corpora["gardening"])
B = counts(corpora["buildapc"])
NA, NB = sum(A.values()), sum(B.values())
vocab = [w for w in (A | B) if (A[w] + B[w]) >= 8]

def log_odds(w):
    pa = (A[w] + 1) / (NA + len(vocab))
    pb = (B[w] + 1) / (NB + len(vocab))
    return np.log(pa / (1 - pa)) - np.log(pb / (1 - pb))

scored = sorted(vocab, key=log_odds)
print("distinctively r/gardening:", scored[-8:][::-1])
print("distinctively r/buildapc: ", scored[:8])
# The She Giggles, He Gallops method, exactly: a smoothed log-odds ratio between two
# corpora. Strongly positive = distinctively A; strongly negative = distinctively B;
# the middle is shared English. The corpus PAIR is a choice: swap in any two.

# The published version (Monroe, Colaresi & Quinn, "Fightin' Words", 2008) adds
# habit 3: standardize each word's log-odds by its variance, "how big is this gap
# relative to the evidence behind it?" One line:
def z_odds(w):
    return log_odds(w) / np.sqrt(1 / (A[w] + 1) + 1 / (B[w] + 1))

scored_z = sorted(vocab, key=z_odds)
print("\nstandardized, r/gardening:", scored_z[-8:][::-1])
print("standardized, r/buildapc: ", scored_z[:8])
# Compare the two lists and notice the character change: topic words sink, and
# quiet, pervasive habits surface. r/buildapc's strongest standardized markers are
# you/your (an advice room); r/gardening's are i/we/them (a stories room). A small
# gap in a very common word can be stronger evidence than a big gap in a rare one,
# and that is exactly what standardizing reveals: register, not just topic. Both
# lists are true; they answer different questions. One more honesty check waits at
# the end of the notebook: rank 1,500 words and SOME look distinctive by pure luck.


## Is the difference real? p-values, effect sizes, and the shuffle test

Your top keyness word is more common in one community than in the other. But small
corpora produce flukes. The honest check needs no formulas: **shuffle the labels and
count again**. If comments were randomly dealt into the two piles a thousand times, how
often would chance alone produce a gap as big as the real one? That fraction has a
famous name: it is a **p-value**, the number in every "p < .05" you will ever read.
Rarely matched: a finding. Often matched: a coincidence. Then two traps and a
confession, below.


In [ ]:
rng = np.random.default_rng(0)
word = scored[-1]                       # our most "distinctively gardening" word
docs_all = corpora["gardening"] + corpora["buildapc"]
word_counts = np.array([tokenize(d).count(word) for d in docs_all])
totals = np.array([max(1, len(tokenize(d))) for d in docs_all])
labels_g = np.array([1] * len(corpora["gardening"]) + [0] * len(corpora["buildapc"]))

def rate_gap(lab):
    return word_counts[lab == 1].sum() / totals[lab == 1].sum() - word_counts[lab == 0].sum() / totals[lab == 0].sum()

observed = rate_gap(labels_g)
shuffled = []
for _ in range(1000):
    rng.shuffle(labels_g)
    shuffled.append(rate_gap(labels_g))

beat = sum(abs(g) >= abs(observed) for g in shuffled)
plt.figure(figsize=(6, 3.5))
plt.hist(shuffled, bins=30)
plt.axvline(observed, color="red", linewidth=2, label=f"the real gap for {word!r}")
plt.title(f"1,000 shuffles: chance matched the real gap {beat} times")
plt.xlabel("gap produced by a random shuffle"); plt.ylabel("shuffles")
plt.legend(); plt.tight_layout(); plt.show()
print(f"chance beat the observed gap in {beat}/1000 shuffles: empirical p = {beat/1000:.3f}",
      "- probably not chance." if beat < 50 else "- could easily be chance; don't lean on it.")

# A confession about this test. We tested scored[-1]: the word we picked BECAUSE its
# gap was the most extreme of ~1,500. Of course it beats the shuffles; the selection
# rigged the race. The clean version states the hypothesis BEFORE looking. Ours,
# written before class: "gardeners say water more." Test exactly that word:
def shuffle_test(word, docs_a, docs_b, n_shuffles=1000):
    wc = np.array([tokenize(d).count(word) for d in docs_a + docs_b])
    tt = np.array([max(1, len(tokenize(d))) for d in docs_a + docs_b])
    lab = np.array([1] * len(docs_a) + [0] * len(docs_b))
    gap = lambda l: wc[l == 1].sum() / tt[l == 1].sum() - wc[l == 0].sum() / tt[l == 0].sum()
    obs = gap(lab)
    r = np.random.default_rng(0)
    beats = 0
    for _ in range(n_shuffles):
        r.shuffle(lab)
        beats += abs(gap(lab)) >= abs(obs)
    return obs, beats / n_shuffles

obs_w, p_w = shuffle_test("water", corpora["gardening"], corpora["buildapc"])
print(f"\npre-registered 'water': gap = {1000*obs_w:+.2f} per 1,000 words, p = {p_w:.3f}")


In [ ]:
# Trap 1: "significant" does not mean "big." The money row from the rates table:
obs_m, p_m = shuffle_test("money", corpora["buildapc"], corpora["relationship_advice"])
print(f"'money', buildapc vs relationship_advice: gap = {1000*obs_m:+.2f} per 1,000 words, p = {p_m:.3f}")

# Two separate questions, always report both:
#   the EFFECT SIZE (how big is the gap?)  and  the p-value (could chance deal it?).
# With enough comments, a microscopic gap turns "significant"; with few, a huge gap
# can fail the test. "Significant" answers "is it chance?", never "does it matter?"
# The sentence to write: "X per 1,000 words more (p = Y)", then argue whether X is
# worth anyone's attention.


In [ ]:
# Trap 2: test enough words and luck looks like discovery. Watch: split ONE
# community's comments randomly in half and run the exact keyness from above.
# There is no real difference here. None.
r = np.random.default_rng(1)
mixed = corpora["gardening"].copy()
r.shuffle(mixed)
half_a, half_b = mixed[:150], mixed[150:]

A2, B2 = counts(half_a), counts(half_b)
NA2, NB2 = sum(A2.values()), sum(B2.values())
vocab2 = [w for w in (A2 | B2) if (A2[w] + B2[w]) >= 8]
def lo2(w):
    pa = (A2[w] + 1) / (NA2 + len(vocab2)); pb = (B2[w] + 1) / (NB2 + len(vocab2))
    return np.log(pa / (1 - pa)) - np.log(pb / (1 - pb))
sc2 = sorted(vocab2, key=lo2)
print('"distinctive" words of random half A:', sc2[-8:][::-1])
print('"distinctive" words of random half B:', sc2[:8])

# Those lists look as real as the gardening/buildapc ones, and they are pure noise.
# Rank 1,500 words and the extremes are impressive BY CONSTRUCTION; at p < .05,
# roughly 75 of 1,500 truly-identical words pass by luck. This is the multiple
# comparisons problem, and it applies upward: Bollen et al. counted 241 phrases.
# The defenses, weakest to strongest: correct the threshold (divide .05 by the number
# of tests: Bonferroni), pre-register the words you will test, or check the winners
# on FRESH data you held out. Week 8 builds the third into your own project.


In [ ]:
# The complement to "is it chance?": "how big is it, give or take?" Resample your own
# comments with replacement 1,000 times and recompute the gap each time: a bootstrap.
# The middle 95% of those gaps is a confidence interval.
wc_g = np.array([tokenize(d).count("water") for d in corpora["gardening"]])
tt_g = np.array([max(1, len(tokenize(d))) for d in corpora["gardening"]])
wc_b = np.array([tokenize(d).count("water") for d in corpora["buildapc"]])
tt_b = np.array([max(1, len(tokenize(d))) for d in corpora["buildapc"]])

r = np.random.default_rng(2)
boot_gaps = []
for _ in range(1000):
    ia = r.integers(0, len(wc_g), len(wc_g))       # resample gardening comments
    ib = r.integers(0, len(wc_b), len(wc_b))       # resample buildapc comments
    boot_gaps.append(1000 * (wc_g[ia].sum() / tt_g[ia].sum() - wc_b[ib].sum() / tt_b[ib].sum()))
lo_ci, hi_ci = np.percentile(boot_gaps, [2.5, 97.5])

plt.figure(figsize=(6, 3.2))
plt.hist(boot_gaps, bins=30, color="#3f6f5f")
plt.axvline(lo_ci, color="#b9852f", linewidth=2); plt.axvline(hi_ci, color="#b9852f", linewidth=2)
plt.title(f'"water" gap: {np.mean(boot_gaps):.1f} per 1,000 words, 95% CI [{lo_ci:.1f}, {hi_ci:.1f}]')
plt.xlabel("gap in resampled data"); plt.ylabel("resamples"); plt.tight_layout(); plt.show()
# The shuffle test asks "could the gap be zero?"; the bootstrap asks "what range of
# sizes is believable?" A finding worth publishing survives both: an interval far
# from zero, and wide enough to keep you humble about the exact number.


## Where counting fails

Counting is the course's foundation, and it has known holes. Name them now; Week 5
exists to patch some of them.


In [ ]:
line = "I am not happy, and this is not a love story."
print("the counter sees:", [w for w in tokenize(line) if w in ("happy", "love")])
# 1. Negation: happy and love count as present; the sentence denies both.
#    (Bigrams catch a little of this: "not happy" is countable. Most context is not.)

try:
    print("2. stemmer collisions:", stem("universal"), stem("university"), stem("universe"))
    print("3. lemmas need context:", lemma("saw"), "the tool, vs", lemma("saw", "v"), "the past of see")
except NameError:
    print("(run the NLTK cell above to see the stemmer and lemma examples)")

# 4. Polysemy: "sick beat" and "sick child" add to the same count. A word type is not
#    a meaning.
# Every method this week inherits all four holes: tf-idf reweights the counts, k-means
# groups by them, keyness compares them. None of them reads. Week 5's embeddings were
# invented for exactly these failures, and they bring their own.


## Playground: your question

Everything above is a kit. Pick a question and answer it with the pieces: choose any two
corpora (or two slices of one), a word or a pattern, and run keyness plus the shuffle
test on it. Write the question first, in one sentence; the cell is yours to edit.

In [ ]:
# MY QUESTION: ...one sentence here...
# Any two piles of real text work. Pull any two communities and rerun the keyness:
GROUP_A, NAME_A = corpora["relationship_advice"], "relationship_advice"
GROUP_B, NAME_B = arctic_sample("AskMen", n=200), "AskMen"     # any subreddit works here

A2, B2 = counts(GROUP_A), counts(GROUP_B)
vocab2 = [w for w in (A2 | B2) if (A2[w] + B2[w]) >= 5]
NA2, NB2 = sum(A2.values()), sum(B2.values())
def lo(w):
    pa = (A2[w] + 1) / (NA2 + len(vocab2)); pb = (B2[w] + 1) / (NB2 + len(vocab2))
    return np.log(pa / (1 - pa)) - np.log(pb / (1 - pb))
sc = sorted(vocab2, key=lo)
print(f"distinctively {NAME_A}:", sc[-8:][::-1])
print(f"distinctively {NAME_B}:", sc[:8])
# Then shuffle-test your most distinctive word with the cell above: swap it into
# `word`, rebuild docs_all and the labels from YOUR two groups, and see whether
# chance can match the gap.


## You made a counter, and you can defend it

You defined what a word is, counted three communities three ways, found the words that make a
voice distinctive with the same method as a famous published study, and shuffle-tested the
gap so you know it isn't chance.

**Sketch (homework):** count something in a text you love; make one chart; write one sentence
naming a choice you made. If your count compares two things, shuffle-test the gap.